In [7]:
from google.colab import files
uploaded = files.upload()

Saving twitter_train_en.csv to twitter_train_en (1).csv


In [12]:

import os
import shutil
import pandas as pd
from comet import download_model, load_from_checkpoint

# Clear the cache for the old model if it exists
old_model_cache_path = "/root/.cache/huggingface/hub/models--Unbabel--wmt20-comet-da"
if os.path.exists(old_model_cache_path):
    shutil.rmtree(old_model_cache_path)
    print(f"Removed old model cache: {old_model_cache_path}")

# Load COMET-QE model
# The model 'Unbabel/wmt22-cometkiwi-da' is not directly supported by comet.download_model.
# Replacing with a commonly supported model like 'Unbabel/wmt20-comet-qe-da' for reference-less QE.
model_path = download_model("Unbabel/wmt20-comet-qe-da")
model = load_from_checkpoint(model_path)

def run_comet_qe(csv_path, source_col="text", mt_col="text_en", sample_size=200, seed=42):
    df = pd.read_csv(csv_path)

    df = df[[source_col, mt_col]].dropna().copy()
    df[source_col] = df[source_col].astype(str).str.strip()
    df[mt_col] = df[mt_col].astype(str).str.strip()
    df = df[(df[source_col] != "") & (df[mt_col] != "")].copy()

    if sample_size is not None and len(df) > sample_size:
        df = df.sample(sample_size, random_state=seed).reset_index(drop=True)

    data = [{"src": s, "mt": t} for s, t in zip(df[source_col], df[mt_col])]

    model_output = model.predict(
        data,
        batch_size=8,
        gpus=1 if __import__("torch").cuda.is_available() else 0
    )

    df["comet_qe"] = model_output.scores

    print("Rows scored:", len(df))
    print("Average COMET-QE:", round(df["comet_qe"].mean(), 4))
    print("Median COMET-QE:", round(df["comet_qe"].median(), 4))
    print("Min COMET-QE:", round(df["comet_qe"].min(), 4))
    print("Max COMET-QE:", round(df["comet_qe"].max(), 4))

    return df

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

hparams.yaml:   0%|          | 0.00/479 [00:00<?, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.3.5 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt20-comet-qe-da/snapshots/2e7ffc84fb67d99cf92506611766463bb9230cfb/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [13]:
twitter_qe = run_comet_qe(
    "twitter_train_en.csv",
    source_col="text",
    mt_col="text_en",
    sample_size=200
)
twitter_qe.head()

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|

Rows scored: 200
Average COMET-QE: 0.1869
Median COMET-QE: 0.1876
Min COMET-QE: -0.6035
Max COMET-QE: 0.5754


,text,text_en,comet_qe
0,नेपाल में पाँच महिना में अर्ब करोड से अधिक मूल...,Imports of medicines and raw materials worth o...,-0.016386
1,आजका ब्रोडसिटका समाचार शीर्षक नेकपा संकट राष्ट...,Today's broadcast news headline is the NECPA c...,-0.094589
2,कोभिड का बिरामीलाई अस्पतालसँग समन्वय गरेर मात्...,Referring only a COVID patient in coordination...,-0.058574
3,नेपालमा कोरोना भाइरस कोभिड को सङ्क्रमण आजको मि...,The coronavirus outbreak in Nepal of COVID-19 ...,0.160773
4,कोभिड का संक्रमित बढ्न थालेपछि पोखरा महानगरको ...,"As the number of Covid cases has increased, at...",0.034648


In [14]:
from google.colab import files
uploaded = files.upload()

Saving youtube_train_en.csv to youtube_train_en.csv


In [15]:
youtube_qe = run_comet_qe(
    "youtube_train_en.csv",
    source_col="text",
    mt_col="text_en",
    sample_size=200
)
youtube_qe.head()

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|

Rows scored: 200
Average COMET-QE: -0.1592
Median COMET-QE: -0.1944
Min COMET-QE: -1.6118
Max COMET-QE: 0.7593


,text,text_en,comet_qe
0,sojha नेपाली हरू लाई आत्याचार गर्ने तिमी हरू क...,Kulat to you who have persecuted sojha Nepalis.,0.076155
1,रास्ट पति को यहि हो बोल्ने सबे थुक यक नारि भयर...,If you tell a woman to marry a ruthless husban...,-0.515138
2,प्रचन्ड ले नै बिगारेको छ अहिले नेपाल भर्स्ट्चा...,"Pradhan has already been corrupted, but even t...",-0.150903
3,यो कुकुर को छाऊरो ले गलत भन्दैछ यो कुकुर भने क...,This dog is a dog that is unemployed and is no...,-0.521581
4,यो पल हाम्रो लागि महत्त्व पुर्ण छ ।,This moment is very important to us.,0.759265


In [16]:
twitter_qe.sort_values("comet_qe").head(10)

,text,text_en,comet_qe
136,कोभिड जुटेर होइन टुटेर सामना गर्रौ,COVID is not a collective disease but a collec...,-0.603529
104,ज्वरो आउँदा बेइजिङ पखाला लाग्दा दिल्ली पेट दुख...,Fever came when Beijing got worse Delhi suffer...,-0.306713
39,कोभिड का कारण थिलो थिलो भए जनता स्थानिय तहले ब...,"If the COVID-19 slowdown is slow, the people w...",-0.302674
168,अनिकालमा भकारि हेर्दै मुसा रुन्छ रे मेरो पनि ह...,Moses cries as I look at the account of the fa...,-0.279612
117,थप जनामा देखियो कोभिड पोजेटिभ कहाँ कति थपिए,See more in the next few days where the COVID ...,-0.265082
177,सालकमा भेटियो कोरोनासँग मिल्दोजुल्दो भाइरस बिह...,A study published in the journal Nature on Thu...,-0.247774
195,नेमकिपा नेता तथा प्रदेश सभा सदस्री का सुरेन्द्...,The head of the Namkipa and a member of the Ra...,-0.209709
105,नेपालमा कोभिड को कारणले पहिलो मृत्यु भएको छ मृ...,The first death in Nepal due to COVID-19 has b...,-0.181718
184,पाल्पा निर्वाचन क्षेत्र नम्वर का संसद दलबहादुर...,The parliament of Palpa constituency of Numwar...,-0.181184
68,जिल्लामा थपिए कोभिड संक्रमित उपत्यकामा मात्र प...,The district is added to the COVID-infected va...,-0.176317


In [17]:
youtube_qe.sort_values("comet_qe").head(10)

,text,text_en,comet_qe
91,हाँ हाँ साले चोर भष्ट् हरू बिनालगानि को उधोग स...,"Yes, yes, yes, yes, yes, yes, yes, yes, yes, y...",-1.611763
78,"माैसम बिग्रएकाे भए किन पठाईयाे त , , , गफ साला...",If the weather is bad and the shipments are no...,-0.793412
177,थुक्क कुकुर नालाएक हरु हो ।,The dog is a dog that is not a dog.,-0.780511
194,ए राडी को बान धमाला पैले तेरी छा ले स्वासनी ला...,A RADI is a brand of the Republic of China.,-0.756994
25,बिप्लव पनिं के हेरेर बसेछः नठोकेर ।,The blind man is looking at the other man: he ...,-0.736778
99,हान न हो सागर जि यो साला को दशै को मा मा बस्नी...,"Han no no sea, who is living in the desert, is...",-0.731954
185,राडि को बान पाखे पत्रकार तै ले सोद्नी प्रसन एस...,Radi is seen as a journalist and has been aske...,-0.725397
29,काङ्रेस चोर हो उधोग नदि नाला सक्यो कम्निस्ट डा...,"Congress was a thief, and the Udhog River was ...",-0.701286
176,कस्ताे चित्न नबुजेकाे गधा हरूकाे कुरा पत्रकार्...,How to talk about a donkey is a more lazy joke...,-0.609621
138,लाज सरम पचे का चोर दलाल हरु ।,The thieves of the Laj Sharma are the brokers.,-0.594035


In [18]:
twitter_qe.to_csv('twitter_comet_qe_results.csv', index=False)
print('twitter_comet_qe_results.csv saved successfully!')

twitter_comet_qe_results.csv saved successfully!


In [19]:
youtube_qe.to_csv('youtube_comet_qe_results.csv', index=False)
print('youtube_comet_qe_results.csv saved successfully!')

youtube_comet_qe_results.csv saved successfully!
